# 301 · Backtesting & Evaluation

Before deploying a new sequential design (like GSD or AVI) to production, you should verify its performance on historical data. This process, called **Backtesting**, allows you to answer questions like:
- "How much earlier would we have stopped on this past experiment?"
- "Would we have made a different decision?"

In this tutorial, we will:
- Load the **ASOS Online-Controlled-Experiment dataset**.
- Run a historical simulation using the unified `backtest()` API.

## 1. Data Preparation

We use the open ASOS digital experiments dataset. We'll pick one experiment and extract incremental batches.

In [ ]:
from pathlib import Path
import ibis
import pandas as pd
from earlysign.core.ledger import Ledger
from earlysign.schema.ES3.Binomial import ArmData
from earlysign.v1.templates.GST_Spending_JennisonTurnbull2000 import (
    JennisonTurnbull2000Template,
)

# Download dataset if needed
data_path = Path("data/asos_digital_experiments_dataset.parquet")
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    !wget -O {data_path} https://osf.io/62t7f/download

df_raw = pd.read_parquet(data_path)
exp_id = "3b4300"
df = df_raw[(df_raw["experiment_id"] == exp_id) & (df_raw["metric_id"] == 1)].copy()
print(f"Loaded {len(df)} monitoring steps for experiment {exp_id}.")

## 2. Defining a Backtest Stream

The `backtest()` API expects a generator of data batches.

In [ ]:
def batch_stream(df):
    df = df.sort_values("time_since_start")
    # Calculate increments between monitoring steps
    dn_c = df["count_c"].diff().fillna(df["count_c"]).astype(int)
    ds_c = (
        (df["count_c"] * df["mean_c"])
        .diff()
        .fillna(df["count_c"] * df["mean_c"])
        .astype(int)
    )
    dn_t = df["count_t"].diff().fillna(df["count_t"]).astype(int)
    ds_t = (
        (df["count_t"] * df["mean_t"])
        .diff()
        .fillna(df["count_t"] * df["mean_t"])
        .astype(int)
    )

    for i in range(len(df)):
        yield [
            ArmData(n=dn_c.iloc[i], success=ds_c.iloc[i], arm="C"),
            ArmData(n=dn_t.iloc[i], success=ds_t.iloc[i], arm="T"),
        ]

## 3. Running the Backtest

We initialize a design (GSD) and run the backtest. The template will process the stream as if it were arriving in real-time.

In [ ]:
ledger = Ledger(ibis.connect("duckdb://:memory:"), "backtest")
ledger.ensure()
trial = JennisonTurnbull2000Template(ledger)

# Design a protocol based on baseline rates from the data
p_baseline = df.iloc[0]["mean_c"]
protocol = trial.design(p_control=p_baseline, p_treatment=p_baseline + 0.005, looks=3)
trial.set_protocol(protocol)

print("Starting backtest...")
results = trial.backtest(batch_stream(df))

print(f"Backtest Final Status: {results['final_status']}")
print(f"Sample Size at Stop: {results['sample_n']:,}")

## 4. Summary

- **Evaluation**: Backtesting provides empirical evidence that a design is suitable for your specific data distribution.
- **Comparison**: You can run multiple backtests with different templates (GSD vs mSPRT) to see which would have performed better.
- **Continuity**: The same code used for backtesting can be used for live orchestration.

In the final tutorial, we will learn how to deploy these designs to **BigQuery** for production usage.